# CPU scoped install and offline compiler only
Set **Accelerator: None**, **Internet: Off** before starting. Attach the fresh receipt-bound transfer bundle and the official competition wheelhouse dataset. Review this notebook before entering the single combined authorization. No starter installation cell, model serving, evaluation or submission is permitted. Preserve previous receipts and outputs; use a fresh session/output directory.

**Every rerun requires Factory reset / a fresh session.** Installs and `/kaggle/working` outputs persist across ordinary cell reruns. Preserve/download old evidence first. A pass with official-environment deviations records compatibility risk; it does not establish that the hidden scorer uses the same image.

B2: use a fresh copy of the official starter for its observed Python 3.12.13 image, but execute only these compiler-only cells. Never run starter serving/evaluation cells. B2 has its own receipt; the M2 receipt cannot be used.


In [ ]:
AUTHORIZATION = ""
EXPECTED = "AUTHORIZE FORGEAI GEMMA4 B2 KAGGLE CPU SCOPED INSTALL AND COMPILER ONLY"
if AUTHORIZATION != EXPECTED:
    raise RuntimeError("Maintainer approval required; no installation or compiler execution")
import importlib.util
import json
from pathlib import Path
import sys
sys.dont_write_bytecode = True
if sys.version_info[:2] != (3, 12) or sys.platform != "linux":
    raise RuntimeError("ENVIRONMENT_BLOCKED: Python 3.12 Linux required")
BUNDLE = Path("/kaggle/input/forgeai-gemma4-b2/bundle")
WHEELHOUSE = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
OUTPUT = Path("/kaggle/working/gemma4-b2-compiler")
LOG = Path("/kaggle/working/gemma4-b2-compiler.log")
if not (BUNDLE / "local-receipt.json").is_file() or not WHEELHOUSE.is_dir():
    raise RuntimeError("Both attached inputs are required")
if OUTPUT.exists() or LOG.exists():
    raise RuntimeError("Preserve existing evidence; fresh output paths required")
ENTRY = BUNDLE / "experiments/gemma4_competition/portable_b2.py"
spec = importlib.util.spec_from_file_location("portable_m2", ENTRY)
portable = importlib.util.module_from_spec(spec)
spec.loader.exec_module(portable)
portable.verify_local_receipt(BUNDLE / "local-receipt.json")


In [ ]:
# The only install cell. Every wheel pin and bundled duplicate is checked first.
# No dependencies/index, no broad starter cell, no .pth cleanup or further installs.
if AUTHORIZATION != EXPECTED:
    raise RuntimeError("Combined approval required")
evidence = portable.scoped_install(AUTHORIZATION, BUNDLE / "local-receipt.json", WHEELHOUSE, OUTPUT)
if evidence["state"] not in portable.INSTALL_PASS_STATES:
    raise RuntimeError("ENVIRONMENT_BLOCKED: preserve install-evidence.json; stop, do not install further")
# Version-equality gate is intentionally AFTER the scoped install.
if portable.installed_versions() != portable.INSTALL_PACKAGES:
    raise RuntimeError("ENVIRONMENT_BLOCKED: post-install version mismatch")


The next cell performs directory validation, full offline compilation with the unchanged closed registries, official negative controls and deterministic ZIP packaging. No model or tool implementation runs. Dependency/source checks remain mandatory. A failure stops the slice; it does not authorize additional installation.


In [ ]:
import os
import subprocess
import tempfile
if AUTHORIZATION != EXPECTED:
    raise RuntimeError("Combined approval required")
portable.verify_install_evidence(OUTPUT / "install-evidence.json", portable.sha((BUNDLE / "local-receipt.json").read_bytes()))
if LOG.exists():
    raise RuntimeError("Preserve previous compiler log")
with tempfile.TemporaryDirectory(prefix="gemma-notebook-home-") as home:
    clean_env, removed = portable.child_environment(dict(os.environ), home)
    print(json.dumps({"removed_environment_names": removed}))
    with LOG.open("x", encoding="utf-8") as log:
        log.write(json.dumps({"removed_environment_names": removed}) + "\n")
        log.flush()
        process = subprocess.run([sys.executable, "-I", "-B", str(ENTRY),
            "--receipt", str(BUNDLE / "local-receipt.json"), "--output", str(OUTPUT)],
            cwd=home, env=clean_env, stdout=log, stderr=subprocess.STDOUT, timeout=300)
print(LOG.read_text(encoding="utf-8"))
if process.returncode:
    raise RuntimeError("Portable compiler failed or environment blocked; preserve evidence and stop")


Download `gemma4-b2-compiler.log` and the complete `gemma4-b2-compiler/` output directory. On success it contains `install-evidence.json`, `compiler-output.json`, `manifest.json`, and `submission.zip`. The manifest includes Python/package versions, installation evidence, sorted ZIP members and ZIP SHA256. If installation fails, download its evidence and stop. Portable PASS alone is not B2 qualification. Reconcile locally against the fresh receipt, then perform the separate pre-commit check. Do not submit the ZIP.


Extract compiler 

In [ ]:
import zipfile, os, glob, hashlib
from IPython.display import FileLink

# Find the receipt uploaded as input and hash it
receipts = glob.glob("/kaggle/input/**/local-receipt.json", recursive=True)
print("receipts found:", receipts)
receipt_hash = hashlib.sha256(open(receipts[0], "rb").read()).hexdigest()
short = receipt_hash[:8]
print("full sha256:", receipt_hash)
print("label:", short)

src_dir = "/kaggle/working/gemma4-b2-compiler"
log = "/kaggle/working/gemma4-b2-compiler.log"
out_name = f"gemma4-b2-compiler-{short}.zip"
out = f"/kaggle/working/{out_name}"

with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
    for name in sorted(os.listdir(src_dir)):
        z.write(os.path.join(src_dir, name), arcname=f"gemma4-b2-compiler/{name}")
    z.write(log, arcname="gemma4-b2-compiler.log")

print(sorted(os.listdir(src_dir)))
FileLink(out_name)